## Задание

- [x] Выбрать датасет
- [x] Определить задачу аппроксимации
- [x] Разбить датасет на обучающую и экспериментальную выборку
- [x] Провести корреляционный анализ
- [x] Выделить 1-2 переменные, которые влияют на выход
- [ ] Разбить переменные на термы (распределить равномерно по графику)
- [ ] Реализовать 4 функции принадлежности и для каждой построить графики
- [ ] Реализовать машину нечёткого вывода (Синглтон / Мамдани / Такаги-Сугено)

## Данные

В качестве датасета возьмём данные об использовании Инстаграм\*: [Social Media User Analysis](https://www.kaggle.com/datasets/rockyt07/social-media-user-analysis/data).
<br/>Будем аппроксимировать уровень счастья пользователя на основе его активности в соцсети.

Разделим датасет на обучающую и экспериментальную выборку в соотношении 70 к 30.

*\* соцсеть Инстаграм признана экстремистской, её деятельность запрещена на территории РФ*

## Корреляционная матрица

Начнём с вычисления корреляции между параметрами. Выберем 3 наиболее коррелирующих с параметром оценки счастья пользователя.

In [3]:
%use dataframe
%use kandy

// read dataset
val rawData = DataFrame.read("data/instagram_usage_lifestyle.csv")
val learnData = rawData.head((rawData.rowsCount() * 0.7).toInt())
val experimentData = rawData.tail((rawData.rowsCount() * 0.3).toInt())
println("""
    Rows total: ${rawData.rowsCount()}
    Rows for learning: ${learnData.rowsCount()}
    Rows for experimental: ${experimentData.rowsCount()}
    """.trimIndent())

// evaluate correlation matrix (using pearson correlation coefficient)
val correlationMatrix = learnData.select { it.all() }.corr()

// select 3 most correlated params for user happiness
val correlatedColumns: List<String> =
    correlationMatrix.first {
        it["column"] == "self_reported_happiness"
    }.let { happinessRow ->
        (happinessRow.columnNames() - "column" - "self_reported_happiness")
            .sortedBy { columnName ->
                (happinessRow[columnName] as Double).absoluteValue
            }.reversed().subList(0, 3)
    }

// show filtered correlation matrix
correlationMatrix.filter {
    it["column"] == "self_reported_happiness"
}.select("column", *correlatedColumns.toTypedArray())

Rows total: 1547896
Rows for learning: 1083527
Rows for experimental: 464368


column,daily_active_minutes_instagram,likes_given_per_day,time_on_feed_per_day
self_reported_happiness,"-0,372625","-0,365501","-0,363294"


Таким образом, `daily_active_minutes_instagram` (активное время в соцсети), `likes_given_per_day` (поставленные лайки) и `time_on_feed_per_day` (время в новостной ленте) имеют заметную отрицательную корреляцию c уровнем счастья пользователя.
<br/>Визуализируем на графике точки и линейную регрессию:
$$\hat{\beta} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

In [4]:
import org.jetbrains.kotlinx.kandy.ir.Plot
import org.jetbrains.kotlinx.kandy.util.color.Color

// draw linear regression
/**
 * Columns must be [Int]
 */
fun displayLinearRegression(
    columnXName: String, columnXTitle: String,
    columnYName: String, columnYTitle: String,
    limit: Int = learnData.rowsCount()
) { // y = kx + b
    val x = learnData[columnXName].cast<Int>().toList()
    val y = learnData[columnYName].cast<Int>().toList()

    val meanX = x.average()
    val meanY = y.average()

    val k = x.zip(y).sumOf { (xi, yi) ->
        (xi - meanX) * (yi - meanY)
    } / x.sumOf {
        (it - meanX).pow(2)
    }
    val b = meanY - k * meanX

    val xLine = listOf(x.min().toDouble(), x.max().toDouble())
    val yLine = xLine.map { k * it + b }

    DISPLAY(learnData.head(limit).plot {
        points {
            x(column<Double>(columnXName)) {
                axis.name = columnXTitle
            }
            y(column<Int>(columnYName)) {
                axis.name = columnYTitle
            }
            color = Color.BLUE
        }
        line {
            x(xLine)
            y(yLine)
            color = Color.RED
        }
    })
}

displayLinearRegression(
    "daily_active_minutes_instagram", "Ежедневное использование (мин)",
    "self_reported_happiness", "Уровень счастья (оценка пользователя)",
    1000
)
displayLinearRegression(
    "likes_given_per_day", "Количество поставленных лайков (в день)",
    "self_reported_happiness", "Уровень счастья (оценка пользователя)",
    1000
)
displayLinearRegression(
    "time_on_feed_per_day", "Ежедневный скролл ленты (мин)",
    "self_reported_happiness", "Уровень счастья (оценка пользователя)",
    1000
)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="lLm2Jd"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("lLm2Jd");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"self_reported_happiness":[8.0,1.0,10.0,1.0,1.0,3.0,10.0,3.0,6.0,10.0,3.0,7.0,10.0,8.0,3.0,10.0,9.0,2.0,3.0,5.0,5.0,8.0,1.0,4.0,10.0,2.0,8.0,9.0,1.0,7.0,2.0,1.0,9.0,8.0,6.0,5.0,7.0,2.0,9.0,6.0,7.0,6.0,4.0,7.0,2.0,1.0,1.0,9.0,6.0,1.0,8.0,6.0,4.0,1.0,5.0,9.0,4.0,9.0,4.0,10.0,8.0,4.0,2.0,1.0,7.0,6.0,8.0,4.0,3.0,8.0,10.0,5.0,2.0,2.0,4.0,7.0,3.0,1.0,6.0,8.0,2.0,4.0,2.0,5.0,5.0,6.0,7.0,4.0,9.0,9.0,1.0,3.0,4.0,6.0,7.0,2.0,5.0,7.0,4.0,4.0,3.0,4.0,6.0,1.0,7.0,10.0,8.0,10.0,8.0,8.0,9.0,6.0,1.0,7.0,6.0,3.0,1.0,10.0,4.0,9.0,7.0,1.0,4.0,10.0,2.0,9.0,9.0,3.0,8.0,8.0,3.0,6.0,1.0,3.0,10.0,4.0,1.0,1.0,10.0,2.0,4.0,5.0,10.0,10.0,8.0,4.0,6.0,5.0,3.0,3.0,2.0,6.0,1.0,7.0,7.0,8.0,6.0,3.0,6.0,3.0,3.0,9.0,6.0,1.0,2.0,9.0,9.0,5.0,9.0,2.0,8.0,4.0,2.0,5.0,1.0,8.0,1.0,9.0,8.0,7.0,5.0,1.0,1.0,9.0,9.0,1.0,8.0,1.0,10.0,6.0,7.0,7.0,7.0,9.0,7.0,10.0,5.0,7.0,9.0,9.0,4.0,3.0,3.0,10.0,3.0,5.0,2.0,9.0,8.0,6.0,3.0,5.0,10.0,8.0,4.0,8.0,9.0,5.0,3.0,2.0,6.0,5.0,2.0,9.0,7.0,6.0,6.0,1.0,10.0,7.0,8.0,6.0,9.0,8.0,2.0,3.0,8.0,6.0,7.0,5.0,7.0,6.0,5.0,5.0,7.0,6.0,3.0,10.0,3.0,8.0,10.0,7.0,5.0,7.0,2.0,1.0,3.0,3.0,4.0,7.0,5.0,3.0,4.0,4.0,4.0,7.0,8.0,7.0,9.0,1.0,4.0,2.0,2.0,7.0,9.0,7.0,9.0,2.0,1.0,6.0,9.0,1.0,2.0,7.0,6.0,2.0,10.0,10.0,9.0,10.0,9.0,6.0,4.0,3.0,5.0,5.0,1.0,1.0,2.0,6.0,9.0,3.0,9.0,10.0,6.0,9.0,6.0,9.0,10.0,5.0,6.0,1.0,6.0,5.0,4.0,9.0,2.0,7.0,7.0,10.0,1.0,6.0,10.0,6.0,2.0,1.0,1.0,1.0,4.0,5.0,2.0,8.0,1.0,2.0,7.0,2.0,5.0,1.0,3.0,2.0,9.0,10.0,4.0,9.0,9.0,8.0,3.0,6.0,4.0,5.0,1.0,6.0,10.0,7.0,6.0,10.0,8.0,3.0,8.0,7.0,6.0,3.0,9.0,3.0,9.0,10.0,9.0,9.0,5.0,2.0,8.0,3.0,1.0,8.0,7.0,7.0,1.0,5.0,1.0,4.0,2.0,2.0,5.0,2.0,6.0,3.0,3.0,7.0,8.0,7.0,8.0,7.0,1.0,2.0,1.0,5.0,8.0,2.0,7.0,8.0,4.0,7.0,6.0,6.0,3.0,10.0,10.0,10.0,6.0,2.0,9.0,8.0,7.0,5.0,9.0,6.0,9.0,3.0,9.0,9.0,8.0,6.0,1.0,1.0,7.0,3.0,4.0,3.0,10.0,6.0,2.0,6.0,9.0,8.0,10.0,2.0,4.0,9.0,8.0,8.0,5.0,6.0,7.0,5.0,9.0,1.0,9.0,10.0,5.0,8.0,4.0,7.0,4.0,4.0,10.0,6.0,4.0,7.0,5.0,4.0,7.0,1.0,3.0,9.0,2.0,4.0,10.0,3.0,5.0,2.0,9.0,1.0,4.0,5.0,6.0,2.0,6.0,2.0,4.0,7.0,1.0,6.0,6.0,8.0,6.0,6.0,2.0,2.0,3.0,8.0,7.0,9.0,10.0,10.0,4.0,8.0,10.0,1.0,5.0,5.0,8.0,5.0,10.0,7.0,7.0,6.0,6.0,3.0,3.0,5.0,6.0,9.0,5.0,7.0,6.0,6.0,1.0,7.0,8.0,5.0,8.0,1.0,1.0,1.0,2.0,1.0,8.0,7.0,9.0,1.0,3.0,5.0,2.0,3.0,2.0,3.0,1.0,8.0,3.0,6.0,6.0,7.0,6.0,6.0,2.0,4.0,4.0,7.0,5.0,9.0,2.0,9.0,8.0,5.0,2.0,8.0,4.0,6.0,7.0,7.0,10.0,9.0,7.0,9.0,9.0,7.0,2.0,9.0,7.0,8.0,10.0,10.0,3.0,2.0,2.0,1.0,5.0,2.0,9.0,1.0,8.0,5.0,4.0,10.0,2.0,8.0,2.0,4.0,1.0,2.0,5.0,2.0,10.0,3.0,9.0,5.0,9.0,8.0,5.0,4.0,6.0,1.0,4.0,10.0,7.0,6.0,4.0,3.0,10.0,1.0,4.0,4.0,9.0,3.0,7.0,5.0,4.0,9.0,2.0,6.0,6.0,3.0,10.0,1.0,7.0,8.0,9.0,1.0,4.0,10.0,2.0,4.0,9.0,3.0,6.0,6.0,3.0,3.0,2.0,8.0,5.0,1.0,4.0,4.0,9.0,5.0,9.0,8.0,8.0,7.0,2.0,7.0,5.0,5.0,5.0,1.0,3.0,10.0,1.0,7.0,3.0,4.0,2.0,4.0,6.0,2.0,3.0,6.0,3.0,6.0,4.0,1.0,1.0,7.0,2.0,1.0,9.0,2.0,7.0,4.0,4.0,4.0,2.0,9.0,10.0,6.0,4.0,2.0,9.0,9.0,2.0,9.0,3.0,5.0,9.0,3.0,7.0,9.0,5.0,9.0,10.0,1.0,4.0,10.0,10.0,9.0,9.0,7.0,6.0,7.0,2.0,10.0,4.0,8.0,7.0,3.0,3.0,2.0,4.0,8.0,7.0,6.0,6.0,2.0,7.0,5.0,8.0,10.0,10.0,9.0,4.0,5.0,8.0,7.0,7.0,6.0,3.0,9.0,9.0,1.

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="oF1jC8"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("oF1jC8");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"self_reported_happiness":[8.0,1.0,10.0,1.0,1.0,3.0,10.0,3.0,6.0,10.0,3.0,7.0,10.0,8.0,3.0,10.0,9.0,2.0,3.0,5.0,5.0,8.0,1.0,4.0,10.0,2.0,8.0,9.0,1.0,7.0,2.0,1.0,9.0,8.0,6.0,5.0,7.0,2.0,9.0,6.0,7.0,6.0,4.0,7.0,2.0,1.0,1.0,9.0,6.0,1.0,8.0,6.0,4.0,1.0,5.0,9.0,4.0,9.0,4.0,10.0,8.0,4.0,2.0,1.0,7.0,6.0,8.0,4.0,3.0,8.0,10.0,5.0,2.0,2.0,4.0,7.0,3.0,1.0,6.0,8.0,2.0,4.0,2.0,5.0,5.0,6.0,7.0,4.0,9.0,9.0,1.0,3.0,4.0,6.0,7.0,2.0,5.0,7.0,4.0,4.0,3.0,4.0,6.0,1.0,7.0,10.0,8.0,10.0,8.0,8.0,9.0,6.0,1.0,7.0,6.0,3.0,1.0,10.0,4.0,9.0,7.0,1.0,4.0,10.0,2.0,9.0,9.0,3.0,8.0,8.0,3.0,6.0,1.0,3.0,10.0,4.0,1.0,1.0,10.0,2.0,4.0,5.0,10.0,10.0,8.0,4.0,6.0,5.0,3.0,3.0,2.0,6.0,1.0,7.0,7.0,8.0,6.0,3.0,6.0,3.0,3.0,9.0,6.0,1.0,2.0,9.0,9.0,5.0,9.0,2.0,8.0,4.0,2.0,5.0,1.0,8.0,1.0,9.0,8.0,7.0,5.0,1.0,1.0,9.0,9.0,1.0,8.0,1.0,10.0,6.0,7.0,7.0,7.0,9.0,7.0,10.0,5.0,7.0,9.0,9.0,4.0,3.0,3.0,10.0,3.0,5.0,2.0,9.0,8.0,6.0,3.0,5.0,10.0,8.0,4.0,8.0,9.0,5.0,3.0,2.0,6.0,5.0,2.0,9.0,7.0,6.0,6.0,1.0,10.0,7.0,8.0,6.0,9.0,8.0,2.0,3.0,8.0,6.0,7.0,5.0,7.0,6.0,5.0,5.0,7.0,6.0,3.0,10.0,3.0,8.0,10.0,7.0,5.0,7.0,2.0,1.0,3.0,3.0,4.0,7.0,5.0,3.0,4.0,4.0,4.0,7.0,8.0,7.0,9.0,1.0,4.0,2.0,2.0,7.0,9.0,7.0,9.0,2.0,1.0,6.0,9.0,1.0,2.0,7.0,6.0,2.0,10.0,10.0,9.0,10.0,9.0,6.0,4.0,3.0,5.0,5.0,1.0,1.0,2.0,6.0,9.0,3.0,9.0,10.0,6.0,9.0,6.0,9.0,10.0,5.0,6.0,1.0,6.0,5.0,4.0,9.0,2.0,7.0,7.0,10.0,1.0,6.0,10.0,6.0,2.0,1.0,1.0,1.0,4.0,5.0,2.0,8.0,1.0,2.0,7.0,2.0,5.0,1.0,3.0,2.0,9.0,10.0,4.0,9.0,9.0,8.0,3.0,6.0,4.0,5.0,1.0,6.0,10.0,7.0,6.0,10.0,8.0,3.0,8.0,7.0,6.0,3.0,9.0,3.0,9.0,10.0,9.0,9.0,5.0,2.0,8.0,3.0,1.0,8.0,7.0,7.0,1.0,5.0,1.0,4.0,2.0,2.0,5.0,2.0,6.0,3.0,3.0,7.0,8.0,7.0,8.0,7.0,1.0,2.0,1.0,5.0,8.0,2.0,7.0,8.0,4.0,7.0,6.0,6.0,3.0,10.0,10.0,10.0,6.0,2.0,9.0,8.0,7.0,5.0,9.0,6.0,9.0,3.0,9.0,9.0,8.0,6.0,1.0,1.0,7.0,3.0,4.0,3.0,10.0,6.0,2.0,6.0,9.0,8.0,10.0,2.0,4.0,9.0,8.0,8.0,5.0,6.0,7.0,5.0,9.0,1.0,9.0,10.0,5.0,8.0,4.0,7.0,4.0,4.0,10.0,6.0,4.0,7.0,5.0,4.0,7.0,1.0,3.0,9.0,2.0,4.0,10.0,3.0,5.0,2.0,9.0,1.0,4.0,5.0,6.0,2.0,6.0,2.0,4.0,7.0,1.0,6.0,6.0,8.0,6.0,6.0,2.0,2.0,3.0,8.0,7.0,9.0,10.0,10.0,4.0,8.0,10.0,1.0,5.0,5.0,8.0,5.0,10.0,7.0,7.0,6.0,6.0,3.0,3.0,5.0,6.0,9.0,5.0,7.0,6.0,6.0,1.0,7.0,8.0,5.0,8.0,1.0,1.0,1.0,2.0,1.0,8.0,7.0,9.0,1.0,3.0,5.0,2.0,3.0,2.0,3.0,1.0,8.0,3.0,6.0,6.0,7.0,6.0,6.0,2.0,4.0,4.0,7.0,5.0,9.0,2.0,9.0,8.0,5.0,2.0,8.0,4.0,6.0,7.0,7.0,10.0,9.0,7.0,9.0,9.0,7.0,2.0,9.0,7.0,8.0,10.0,10.0,3.0,2.0,2.0,1.0,5.0,2.0,9.0,1.0,8.0,5.0,4.0,10.0,2.0,8.0,2.0,4.0,1.0,2.0,5.0,2.0,10.0,3.0,9.0,5.0,9.0,8.0,5.0,4.0,6.0,1.0,4.0,10.0,7.0,6.0,4.0,3.0,10.0,1.0,4.0,4.0,9.0,3.0,7.0,5.0,4.0,9.0,2.0,6.0,6.0,3.0,10.0,1.0,7.0,8.0,9.0,1.0,4.0,10.0,2.0,4.0,9.0,3.0,6.0,6.0,3.0,3.0,2.0,8.0,5.0,1.0,4.0,4.0,9.0,5.0,9.0,8.0,8.0,7.0,2.0,7.0,5.0,5.0,5.0,1.0,3.0,10.0,1.0,7.0,3.0,4.0,2.0,4.0,6.0,2.0,3.0,6.0,3.0,6.0,4.0,1.0,1.0,7.0,2.0,1.0,9.0,2.0,7.0,4.0,4.0,4.0,2.0,9.0,10.0,6.0,4.0,2.0,9.0,9.0,2.0,9.0,3.0,5.0,9.0,3.0,7.0,9.0,5.0,9.0,10.0,1.0,4.0,10.0,10.0,9.0,9.0,7.0,6.0,7.0,2.0,10.0,4.0,8.0,7.0,3.0,3.0,2.0,4.0,8.0,7.0,6.0,6.0,2.0,7.0,5.0,8.0,10.0,10.0,9.0,4.0,5.0,8.0,7.0,7.0,6.0,3.0,9.0,9.0,1.

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="oHdf7Z"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("oHdf7Z");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
"time_on_feed_per_day":[2.0,31.0,3.0,108.0,78.0,29.0,64.0,167.0,64.0,115.0,113.0,68.0,3.0,2.0,172.0,3.0,157.0,256.0,102.0,43.0,83.0,2.0,81.0,107.0,26.0,121.0,132.0,3.0,160.0,112.0,36.0,140.0,47.0,81.0,39.0,73.0,16.0,87.0,19.0,3.0,102.0,39.0,140.0,108.0,3.0,176.0,77.0,13.0,102.0,243.0,121.0,174.0,148.0,97.0,143.0,98.0,50.0,12.0,102.0,44.0,96.0,161.0,118.0,69.0,37.0,2.0,130.0,120.0,119.0,48.0,91.0,67.0,155.0,175.0,94.0,68.0,167.0,128.0,192.0,45.0,175.0,55.0,225.0,2.0,42.0,102.0,9.0,19.0,123.0,19.0,78.0,142.0,90.0,2.0,82.0,109.0,3.0,122.0,57.0,100.0,124.0,170.0,87.0,215.0,140.0,108.0,14.0,38.0,161.0,107.0,68.0,119.0,153.0,171.0,124.0,195.0,78.0,105.0,55.0,63.0,163.0,124.0,51.0,32.0,72.0,64.0,60.0,127.0,126.0,29.0,137.0,124.0,129.0,188.0,64.0,131.0,148.0,184.0,66.0,55.0,35.0,132.0,72.0,89.0,103.0,24.0,99.0,82.0,132.0,67.0,124.0,135.0,105.0,161.0,173.0,115.0,121.0,70.0,6.0,142.0,27.0,7.0,77.0,147.0,86.0,59.0,56.0,154.0,94.0,17.0,100.0,164.0,107.0,45.0,43.0,2.0,123.0,69.0,143.0,50.0,87.0,136.0,187.0,177.0,3.0,27.0,38.0,160.0,73.0,75.0,75.0,63.0,126.0,155.0,182.0,127.0,46.0,136.0,3.0,61.0,110.0,111.0,241.0,37.0,101.0,124.0,192.0,153.0,151.0,48.0,98.0,82.0,10.0,135.0,140.0,108.0,121.0,187.0,164.0,133.0,2.0,81.0,101.0,3.0,64.0,139.0,68.0,173.0,103.0,22.0,75.0,110.0,2.0,39.0,126.0,207.0,113.0,2.0,166.0,84.0,167.0,26.0,179.0,22.0,3.0,92.0,163.0,76.0,139.0,143.0,90.0,45.0,19.0,88.0,4.0,106.0,137.0,155.0,67.0,97.0,196.0,90.0,107.0,43.0,69.0,36.0,21.0,122.0,74.0,135.0,167.0,192.0,73.0,179.0,87.0,93.0,163.0,130.0,168.0,3.0,123.0,149.0,181.0,107.0,152.0,39.0,156.0,18.0,3.0,30.0,51.0,118.0,32.0,86.0,89.0,92.0,89.0,83.0,150.0,3.0,99.0,147.0,3.0,3.0,65.0,33.0,187.0,150.0,2.0,112.0,104.0,63.0,64.0,121.0,58.0,11.0,57.0,112.0,61.0,121.0,59.0,98.0,78.0,32.0,24.0,81.0,106.0,81.0,205.0,59.0,127.0,2.0,89.0,54.0,89.0,133.0,44.0,173.0,248.0,149.0,50.0,52.0,68.0,124.0,24.0,3.0,210.0,153.0,74.0,134.0,255.0,50.0,70.0,50.0,82.0,32.0,197.0,83.0,188.0,2.0,68.0,35.0,25.0,107.0,11.0,89.0,140.0,131.0,156.0,27.0,141.0,89.0,59.0,147.0,143.0,28.0,100.0,71.0,128.0,22.0,48.0,86.0,33.0,224.0,17.0,46.0,166.0,2.0,104.0,145.0,148.0,78.0,217.0,169.0,171.0,66.0,32.0,140.0,123.0,109.0,155.0,2.0,85.0,203.0,194.0,7.0,100.0,8.0,120.0,111.0,29.0,32.0,108.0,30.0,130.0,72.0,147.0,107.0,170.0,3.0,116.0,133.0,139.0,158.0,78.0,24.0,71.0,73.0,134.0,171.0,54.0,14.0,148.0,32.0,146.0,118.0,126.0,2.0,60.0,54.0,160.0,126.0,101.0,137.0,78.0,107.0,37.0,3.0,92.0,96.0,53.0,53.0,198.0,123.0,47.0,47.0,18.0,87.0,148.0,57.0,65.0,138.0,124.0,18.0,55.0,116.0,108.0,170.0,236.0,62.0,2.0,185.0,99.0,24.0,109.0,57.0,66.0,66.0,146.0,75.0,45.0,22.0,135.0,140.0,161.0,154.0,16.0,88.0,127.0,60.0,3.0,59.0,51.0,29.0,171.0,85.0,21.0,268.0,139.0,158.0,43.0,95.0,3.0,115.0,49.0,14.0,3.0,186.0,52.0,65.0,2.0,78.0,84.0,79.0,127.0,104.0,103.0,88.0,2.0,152.0,149.0,108.0,167.0,228.0,197.0,63.0,102.0,156.0,84.0,87.0,172.0,25.0,157.0,46.0,162.0,115.0,107.0,21.0,104.0,56.0,101.0,113.0,72.0,29.0,130.0,209.0,171.0,120.0,142.0,19.0,49.0,72.0,60.0,12.0,176.0,35.0,176.0,39.0,71.0,129.0,160.0,162

Распределим параметры равномерно по термам. По заданию требуется определить все 4 функции принадлежности, поэтому для каждого из 4 параметров (3 входных и 1 выходного) используем разные функции.

Для ежедневного использования соцсети используем **треугольную**:
$$
% Треугольная (a, b, c — левая граница, вершина, правая граница)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x \leq b \\
\frac{c - x}{c - b}, & b < x < c \\
0, & x \geq c
\end{cases}
$$

Для количества поставленных лайков - **трапецеидальную**:
$$
% Трапецеидальная (a, b, c, d — границы и плато)
\mu(x) = \begin{cases}
0, & x \leq a \\
\frac{x - a}{b - a}, & a < x < b \\
1, & b \leq x \leq c \\
\frac{d - x}{d - c}, & c < x < d \\
0, & x \geq d
\end{cases}
$$

Для ежедневного скролла ленты - **параболическую**:
$$
% Параболическая (a, b — границы)
\mu(x) = \begin{cases}
0, & x \leq a \\
1 - \left(\frac{x - b}{b - a}\right)^2, & a < x \leq b \\
1 - \left(\frac{x - b}{c - b}\right)^2, & b < x < c \\
0, & x \geq c
\end{cases}
$$

Для оценки уровня счастья пользователя - **Гаусса**:
$$
% Гауссова (c — центр, σ — ширина)
\mu(x) = e^{-\frac{(x - c)^2}{2\sigma^2}}
$$

In [57]:
/* API */

fun interface TermChartBuilder {
    /**
     * [fromX], [toX] - both inclusive
     */
    fun append(
        // input
        fromX: Double,
        toX: Double,
        termName: String,
        // output
        bufferX: MutableList<Double>,
        bufferY: MutableList<Double>,
        bufferTerm: MutableList<String>
    )
}

fun displayTerms(
    minToMax: Pair<Int, Int>,
    terms: List<String>,
    title: String,
    builder: TermChartBuilder
) {
    val step = (minToMax.second - minToMax.first) / (terms.size.toDouble() - 1) * 2
    val fromX = minToMax.first - step / 2

    val bufferX = ArrayList<Double>()
    val bufferY = ArrayList<Double>()
    val bufferTerm = ArrayList<String>()

    terms.forEachIndexed { i, termName ->
        builder.append(
            fromX + step / 2 * i, fromX + step * (i / 2.0 + 1), termName,
            bufferX, bufferY, bufferTerm
        )
    }

    DISPLAY(learnData.plot {
        layout.title = title
        x.axis.limits = minToMax.first.toDouble() ..  minToMax.second.toDouble()
        y.axis.limits = 0 .. 1
        line {
            x(bufferX)
            y(bufferY)
            color(bufferTerm) { legend.name = "Терм" }
        }
    })
}

/* Term functions */

fun appendTriangularTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(fromX, fromX + (toX - fromX) / 2.0, toX)
    bufferY += listOf(0.0, 1.0, 0.0)
    bufferTerm += List(3) { termName }
}

fun appendTrapezoidalTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    bufferX += listOf(
        fromX,
        fromX + (toX - fromX) / 3.0,
        fromX + (toX - fromX) / 3.0 * 2,
        toX
    )
    bufferY += listOf(0.0, 1.0, 1.0, 0.0)
    bufferTerm += List(4) { termName }
}

fun appendParabolicTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map { 1 - ((it - centerX) / (centerX - fromX)).pow(2) }
    bufferTerm += List(100) { termName }
}

fun appendGaussianTerm(
    // input
    fromX: Double,
    toX: Double,
    termName: String,
    // output
    bufferX: MutableList<Double>,
    bufferY: MutableList<Double>,
    bufferTerm: MutableList<String>
) {
    val valuesX = (0 until 100).map { fromX + (toX - fromX) / 100 * it }
    val centerX = (toX - fromX) / 2 + fromX
    bufferX += valuesX
    bufferY += valuesX.map {
        val sigma = (centerX - fromX) / 3 // (centerX - fromX) is too big
        val numerator = - (it - centerX).pow(2)
        val denominator = 2 * sigma.pow(2)
        exp(numerator / denominator)
    }
    bufferTerm += List(100) { termName }
}

/* Impl */

displayTerms(
    minToMax = learnData["daily_active_minutes_instagram"]
        .cast<Int>()
        .toList()
        .run { min() to max() },
    terms = listOf("немного", "много", "слишком много", "так много, что слов нет"),
    title = "Ежедневное использование соцсети (мин)",
    builder = ::appendTriangularTerm
)

displayTerms(
    minToMax = learnData["likes_given_per_day"]
        .cast<Int>()
        .toList()
        .run { min() to max() },
    terms = listOf("чуть-чуть", "немного", "не очень много", "много", "очень много"),
    title = "Лайков поставлено (в день)",
    builder = ::appendTrapezoidalTerm
)

displayTerms(
    minToMax = learnData["time_on_feed_per_day"]
        .cast<Int>()
        .toList()
        .run { min() to max() },
    terms = listOf("недолго", "довольно долго", "крайне долго"),
    title = "Ежедневный скролл ленты (мин)",
    builder = ::appendParabolicTerm
)

displayTerms(
    minToMax = learnData["self_reported_happiness"]
        .cast<Int>()
        .toList()
        .run { min() to max() },
    terms = listOf("Ужасно", "Плохо", "Нормально", "Хорошо", "Замечательно"),
    title = "Оценка уровня счастья пользователем",
    builder = ::appendGaussianTerm
)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="BDAxHq"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Ежедневное использование соцсети (мин)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[5.0,580.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["немного","немного","немного","много","много","много","слишком много","слишком много","слишком много","так много, что слов нет","так много, что слов нет","так много, что слов нет"],
"x":[-186.66666666666666,5.0,196.66666666666666,5.0,196.66666666666669,388.33333333333337,196.66666666666666,388.33333333333337,580.0,388.33333333333337,580.0,771.6666666666666],
"y":[0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"221"
};
 var containerDiv = document.getElementById("BDAxHq");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Ежедневное использование соцсети (мин) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 немного 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 слишком много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 так много, что слов нет

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="virW2i"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Лайков поставлено (в день)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[8.0,350.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["чуть-чуть","чуть-чуть","чуть-чуть","чуть-чуть","немного","немного","немного","немного","не очень много","не очень много","не очень много","не очень много","много","много","много","много","очень много","очень много","очень много","очень много"],
"x":[-77.5,-20.5,36.5,93.5,8.0,65.0,122.0,179.0,93.5,150.5,207.5,264.5,179.0,236.0,293.0,350.0,264.5,321.5,378.5,435.5],
"y":[0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"color"
}]
}
}],
"spec_id":"224"
};
 var containerDiv = document.getElementById("virW2i");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 Лайков поставлено (в день) 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 Терм 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 чуть-чуть 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 немного 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 не очень много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 много 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 очень много

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="TSwsZJ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Ежедневный скролл ленты (мин)"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[2.0,328.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","недолго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","довольно долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне долго","крайне до

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="euECLW"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Оценка уровня счастья пользователем"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"xlim":[1.0,10.0],
"flip":false,
"ylim":[0.0,1.0]
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true,
"name":"Терм"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"color":"color"
},
"stat":"identity",
"data":{
"color":["Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Ужасно","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Плохо","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Нормально","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хорошо","Хо